In [16]:
import os
import open3d as o3d
import cv2
import numpy as np

# Depth camera parameters:
FX_DEPTH = 5.8262448167737955e+02
FY_DEPTH = 5.8269103270988637e+02
CX_DEPTH = 3.1304475870804731e+02
CY_DEPTH = 2.3844389626620386e+02

def depth_to_pcd(depth_image):
    pcd = []
    height, width = depth_image.shape
    for i in range(height):
        for j in range(width):
            z = depth_image[i][j]
            x = (j - CX_DEPTH) * z / FX_DEPTH
            y = (i - CY_DEPTH) * z / FY_DEPTH
            pcd.append([x, y, z])
    pcd_o3d = o3d.geometry.PointCloud()  # create point cloud object
    pcd_o3d.points = o3d.utility.Vector3dVector(pcd)  # set pcd_np as the point cloud points
    
    
    return pcd_o3d

def rgbd_to_pcd(depth_path, rgb_path):
    color_raw = o3d.io.read_image(rgb_path)
    depth_raw = o3d.io.read_image(depth_path)
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(color_raw, depth_raw, convert_rgb_to_intensity=False)
    print(rgbd_image)
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(
        rgbd_image, 
        o3d.camera.PinholeCameraIntrinsic(o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
        )
    # Flip it, otherwise the pointcloud will be upside down
    pcd.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]])
    return pcd
    
    




In [17]:
def test_myself():
    depth_path = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1/kitchen_small_1_70_depth.png"
    out_path = "/remote-home/liuym/data/test/"
    depth_img = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    pcd = depth_to_pcd(depth_img)
    #save pcd
    o3d.io.write_point_cloud(os.path.join(out_path, "kitchen_small_1_70.ply"), pcd)
    
def test_o3d():
    depth_path = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1/kitchen_small_1_70_depth.png"
    rgb_path = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1/kitchen_small_1_70.png"
    out_path = "/remote-home/liuym/data/test/"
    
    pcd = rgbd_to_pcd(depth_path, rgb_path)
    # colors = np.asarray(pcd.colors)
    # pcd.colors = o3d.utility.Vector3dVector(colors.astype(float) / 255.0)
    o3d.io.write_point_cloud(os.path.join(out_path, "kitchen_small_1_70_colored.ply"), pcd, write_ascii=True)

In [18]:
test_o3d()

RGBDImage of size 
Color image : 640x480, with 3 channels.
Depth image : 640x480, with 1 channels.
Use numpy.asarray to access buffer data.


In [4]:
test_myself()

In [7]:
import open3d as o3d
import numpy as np


def load_rgbd(data):
    color = o3d.io.read_image(data['color_path'])
    depth = o3d.io.read_image(data['depth_path'])
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(color, depth, depth_trunc=4.0, convert_rgb_to_intensity=False)
    return rgbd_image

def rgbd_odometry(source_rgbd_image, target_rgbd_image, odo_init=None):
    pinhole_camera_intrinsic = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)
    option = o3d.pipelines.odometry.OdometryOption()
    if odo_init is None:
        odo_init = np.identity(4)
    print(option)
    
    success_color_term, trans_color_term, info = o3d.pipelines.odometry.compute_rgbd_odometry(
        source_rgbd_image, target_rgbd_image,
        pinhole_camera_intrinsic, odo_init,
        o3d.pipelines.odometry.RGBDOdometryJacobianFromColorTerm(), option
        )
    
    if success_color_term:
        print("Using RGB-D Odometry")
        return {'transform': trans_color_term, 'info':info}
    
    success_hybrid_term, trans_hybrid_term, info = o3d.pipelines.odometry.compute_rgbd_odometry(
        source_rgbd_image, target_rgbd_image,
        pinhole_camera_intrinsic, odo_init,
        o3d.pipelines.odometry.RGBDOdometryJacobianFromHybridTerm(), option
        )
    
    if success_hybrid_term:
        print("Using RGB-D Odometry")
        return {'transform': trans_hybrid_term, 'info':info}
    
    return None

In [8]:
import os

data_dir = "/remote-home/liuym/data/rgbd-scenes/kitchen_small/kitchen_small_1"
rgbd_file_list = ["kitchen_small_1_70", "kitchen_small_1_72", "kitchen_small_1_74", "kitchen_small_1_76"]
rgbd_data_list = []

for rgbd_file in rgbd_file_list:
    data={
            'color_path': os.path.join(data_dir, rgbd_file + ".png"),
            'depth_path': os.path.join(data_dir, rgbd_file + "_depth.png")
        }
    data['rgbd'] = load_rgbd(data)
    rgbd_data_list.append(data)

odo_init = None
camera_poses = [np.identity(4)]
for i in range(len(rgbd_data_list)-1):
    j = (i+1) % len(rgbd_data_list)
    src = rgbd_data_list[i]['rgbd']
    tgt = rgbd_data_list[j]['rgbd']
    
    odo_init = camera_poses[-1]
    odometry = rgbd_odometry(src, tgt, odo_init)
    camera_poses.append(odometry['transform'])
    


OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
depth_diff_max = 0.030000
depth_min = 0.000000
depth_max = 4.000000
Using RGB-D Odometry
OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
depth_diff_max = 0.030000
depth_min = 0.000000
depth_max = 4.000000
Using RGB-D Odometry
OdometryOption class.
iteration_number_per_pyramid_level = [ 20, 10, 5, ] 
depth_diff_max = 0.030000
depth_min = 0.000000
depth_max = 4.000000
Using RGB-D Odometry


In [9]:
camera_poses[0].dtype

dtype('float64')

In [10]:
volume = o3d.pipelines.integration.UniformTSDFVolume(
    length=4.0,
    resolution=512,
    sdf_trunc=0.04,
    color_type=o3d.pipelines.integration.TSDFVolumeColorType.RGB8,
)

camera_intrinsic = o3d.camera.PinholeCameraIntrinsic(
        o3d.camera.PinholeCameraIntrinsicParameters.PrimeSenseDefault)

for i in range(len(camera_poses)):
    print("Integrate {:d}-th image into the volume.".format(i))
    rgbd = rgbd_data_list[i]['rgbd']
    volume.integrate(
        rgbd,
        camera_intrinsic,
        np.linalg.inv(camera_poses[i]),
    )
    
# export
out_path = "/remote-home/liuym/data/test_odo/kitchen_small"









Integrate 0-th image into the volume.
Integrate 1-th image into the volume.
Integrate 2-th image into the volume.
Integrate 3-th image into the volume.


In [11]:
print("Extract triangle mesh")
mesh = volume.extract_triangle_mesh()
mesh.compute_vertex_normals()
o3d.io.write_triangle_mesh(os.path.join(out_path, "kitchen_small_mesh.ply"), mesh, write_vertex_colors=True)

Extract triangle mesh


True

In [12]:
print("Extract voxel-aligned debugging point cloud")
voxel_pcd = volume.extract_voxel_point_cloud()
o3d.io.write_point_cloud(os.path.join(out_path, "kitchen_small_aligned_pc.ply"), voxel_pcd, write_ascii=True)

Extract voxel-aligned debugging point cloud


True

In [ ]:
voxel_grid = volume.extract_voxel_grid()
o3d.io.write_voxel_grid()
o3d.io.write_(os.path.join(out_path, "kitchen_small_voxel.ply"), voxel_grid, write_ascii=True)